In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

!pip install pandas numpy scikit-learn -q

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pickle

DATASET_DIR = "/content/drive/MyDrive/MSIL_Dataset/"
SAVE_DIR    = "/content/drive/MyDrive/MSIL_Dataset/parameters_nslkdd/"
os.makedirs(SAVE_DIR, exist_ok=True)

TRAIN_FILE  = "KDDTrain+.csv"
TEST_FILE   = "KDDTest+.csv"     # used in Cell 9 to build stimulus.mem

N_COMPONENTS   = 4     # PCA components to keep — increase if variance < 85%
THRESHOLD_PCT  = 90     # percentile for anomaly threshold on benign training data
                        # 90 = low threshold → higher DR, slightly higher FAR
                        # 99 = high threshold → lower DR, near-zero FAR
                        # Start at 90, tune via Phase 2 threshold sweep

CLIP_VALUE  = 5.0       # clip scaled features to [-CLIP_VALUE, +CLIP_VALUE]
                        # removes outlier benign samples that inflate threshold

FRAC_BITS   = 12
TOTAL_BITS  = 16

# NSL-KDD features chosen for hardware feasibility + DoS/Probe discriminability
# All map to counters or registers in the FEM — no division needed
FEATURES = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "serror_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_serror_rate",
    "same_srv_rate",
    "diff_srv_rate",          # ← new: % connections to different services (probe signal)
    "dst_host_diff_srv_rate", # ← new: destination host different-service rate
]
# ============================================================

In [ ]:
def parse_label(raw_label):
    """
    NSL-KDD label column: "normal" = benign, everything else = attack.
    Handles trailing whitespace, newlines, and mixed case robustly.
    Returns "BENIGN" or "ATTACK".
    """
    cleaned = str(raw_label).strip().lower()
    return "BENIGN" if cleaned == "normal" else "ATTACK"

In [ ]:
train_path = os.path.join(DATASET_DIR, TRAIN_FILE)
print(f"📂 Loading {TRAIN_FILE} ...")

df_train = pd.read_csv(train_path)
df_train.columns = df_train.columns.str.strip()

# Confirm label column exists
assert "label" in df_train.columns, (
    f"'label' column not found. Columns are: {df_train.columns.tolist()}\n"
    "Make sure you ran the txt→csv conversion with the correct 42-column header."
)

# Parse labels robustly
df_train["parsed_label"] = df_train["label"].apply(parse_label)

label_counts = df_train["parsed_label"].value_counts()
print(f"\nLabel distribution in training file:")
print(f"  BENIGN : {label_counts.get('BENIGN', 0)}")
print(f"  ATTACK : {label_counts.get('ATTACK', 0)}")

# Keep only BENIGN rows for PCA training
benign_df = df_train[df_train["parsed_label"] == "BENIGN"].copy()
print(f"\n✅ Benign rows selected for PCA training: {len(benign_df)}")

if len(benign_df) == 0:
    raise ValueError(
        "No BENIGN rows found. Check your label column. "
        "Print df_train['label'].unique() to see actual values."
    )


📂 Loading KDDTrain+.csv ...

Label distribution in training file:
  BENIGN : 67343
  ATTACK : 58630

✅ Benign rows selected for PCA training: 67343


In [ ]:
missing = [f for f in FEATURES if f not in benign_df.columns]
if missing:
    raise ValueError(f"These features are missing from your CSV: {missing}\n"
                     f"Available columns: {benign_df.columns.tolist()}")

X = benign_df[FEATURES].values.astype(np.float64)

# Remove NaN / inf rows
bad = np.isnan(X).any(axis=1) | np.isinf(X).any(axis=1)
print(f"\nRows with NaN/inf removed: {bad.sum()}")
X_clean = X[~bad]
print(f"Clean benign rows for fitting: {X_clean.shape[0]}")

# Fit StandardScaler
scaler = StandardScaler().fit(X_clean)
Xs = scaler.transform(X_clean)

# Clip outliers — prevents extreme benign samples from inflating the threshold
# This is critical: without clipping the 99th percentile ends up far too high
Xs = np.clip(Xs, -CLIP_VALUE, CLIP_VALUE)

print(f"\nScaled + clipped feature ranges (should be within ±{CLIP_VALUE}):")
for i, feat in enumerate(FEATURES):
    print(f"  {feat:<30} min={Xs[:,i].min():6.2f}  max={Xs[:,i].max():6.2f}")



Rows with NaN/inf removed: 0
Clean benign rows for fitting: 67343

Scaled + clipped feature ranges (should be within ±5.0):
  duration                       min= -0.13  max=  5.00
  src_bytes                      min= -0.03  max=  5.00
  dst_bytes                      min= -0.07  max=  5.00
  count                          min= -0.42  max=  5.00
  srv_count                      min= -0.46  max=  5.00
  serror_rate                    min= -0.14  max=  5.00
  dst_host_count                 min= -1.45  max=  1.06
  dst_host_srv_count             min= -2.05  max=  0.70
  dst_host_serror_rate           min= -0.15  max=  5.00
  same_srv_rate                  min= -5.00  max=  0.21
  diff_srv_rate                  min= -0.20  max=  5.00
  dst_host_diff_srv_rate         min= -0.31  max=  5.00


In [ ]:
pca = PCA(n_components=N_COMPONENTS).fit(Xs)

explained   = pca.explained_variance_ratio_
cumulative  = np.cumsum(explained)

print("\n📊 PCA Explained Variance:")
for i, (ev, cv) in enumerate(zip(explained, cumulative)):
    bar = "█" * int(ev * 60)
    print(f"  PC{i+1}: {ev:.4f}  cumulative={cv:.4f}  {bar}")

print(f"\n✅ Total variance captured: {cumulative[-1]:.4f}")
if cumulative[-1] < 0.85:
    print("⚠️  Below 85% — increase N_COMPONENTS in CELL 1 and re-run from Cell 5.")
elif cumulative[-1] > 0.97:
    print("💡 Above 97% — you could reduce N_COMPONENTS to save RTL hardware cost.")


📊 PCA Explained Variance:
  PC1: 0.2924  cumulative=0.2924  █████████████████
  PC2: 0.2340  cumulative=0.5264  ██████████████
  PC3: 0.1298  cumulative=0.6562  ███████
  PC4: 0.1010  cumulative=0.7572  ██████

✅ Total variance captured: 0.7572
⚠️  Below 85% — increase N_COMPONENTS in CELL 1 and re-run from Cell 5.


In [ ]:
Y_train     = pca.transform(Xs)
X_hat_train = pca.inverse_transform(Y_train)
recon_err   = np.sum((Xs - X_hat_train) ** 2, axis=1)

threshold   = np.percentile(recon_err, THRESHOLD_PCT)

print(f"\n📊 Reconstruction error on benign training data:")
print(f"   Min          = {recon_err.min():.4f}")
print(f"   Mean         = {recon_err.mean():.4f}")
print(f"   Max          = {recon_err.max():.4f}")
print(f"   {THRESHOLD_PCT}th percentile (threshold) = {threshold:.4f}")
print(f"\n   → {100 - THRESHOLD_PCT}% of benign training samples will be flagged as false alarms")
print(f"     at this threshold. Tune THRESHOLD_PCT in CELL 1 to adjust DR vs FAR trade-off.")


📊 Reconstruction error on benign training data:
   Min          = 0.0163
   Mean         = 1.8652
   Max          = 86.4674
   90th percentile (threshold) = 3.2230

   → 10% of benign training samples will be flagged as false alarms
     at this threshold. Tune THRESHOLD_PCT in CELL 1 to adjust DR vs FAR trade-off.


In [ ]:
def to_fixed(val: float) -> int:
    """Q4.12 signed fixed-point, 16-bit two's complement."""
    scaled = int(round(float(val) * (1 << FRAC_BITS)))
    return scaled & ((1 << TOTAL_BITS) - 1)

def write_mem(fpath, values):
    """Write hex values one per line for Verilog $readmemh."""
    with open(fpath, "w") as f:
        for v in values:
            f.write(f"{to_fixed(v):04x}\n")

write_mem(os.path.join(SAVE_DIR, "mean.mem"),      scaler.mean_)
write_mem(os.path.join(SAVE_DIR, "invstd.mem"),    1.0 / scaler.scale_)
write_mem(os.path.join(SAVE_DIR, "pca_w.mem"),     pca.components_.flatten())
write_mem(os.path.join(SAVE_DIR, "threshold.mem"), [threshold])

print("✅ Fixed-point .mem files saved:")
for name in ["mean.mem", "invstd.mem", "pca_w.mem", "threshold.mem"]:
    fpath = os.path.join(SAVE_DIR, name)
    print(f"  {name}  ({os.path.getsize(fpath)} bytes)")

✅ Fixed-point .mem files saved:
  mean.mem  (60 bytes)
  invstd.mem  (60 bytes)
  pca_w.mem  (240 bytes)
  threshold.mem  (5 bytes)


In [ ]:
with open(os.path.join(SAVE_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os.path.join(SAVE_DIR, "pca.pkl"), "wb") as f:
    pickle.dump(pca, f)

with open(os.path.join(SAVE_DIR, "threshold.txt"), "w") as f:
    f.write(str(threshold))

with open(os.path.join(SAVE_DIR, "clip_value.txt"), "w") as f:
    f.write(str(CLIP_VALUE))

with open(os.path.join(SAVE_DIR, "features.txt"), "w") as f:
    for feat in FEATURES:
        f.write(feat + "\n")

print("✅ Model files saved to:", SAVE_DIR)
for name in ["scaler.pkl", "pca.pkl", "threshold.txt", "clip_value.txt", "features.txt"]:
    print(f"  {name}")


✅ Model files saved to: /content/drive/MyDrive/MSIL_Dataset/parameters_nslkdd/
  scaler.pkl
  pca.pkl
  threshold.txt
  clip_value.txt
  features.txt


In [ ]:
test_path = os.path.join(DATASET_DIR, TEST_FILE)
print(f"\n📂 Building Verilog stimulus from: {TEST_FILE}")

df_test_stim  = pd.read_csv(test_path)
df_test_stim.columns = df_test_stim.columns.str.strip()
df_test_stim["parsed_label"] = df_test_stim["label"].apply(parse_label)

X_stim_raw  = df_test_stim[FEATURES].values.astype(np.float64)
labels_stim = df_test_stim["parsed_label"].values

bad_stim    = np.isnan(X_stim_raw).any(axis=1) | np.isinf(X_stim_raw).any(axis=1)
X_stim_c    = X_stim_raw[~bad_stim]
labels_c    = labels_stim[~bad_stim]

# Apply same scaler + clip as training
X_stim_sc   = np.clip(scaler.transform(X_stim_c), -CLIP_VALUE, CLIP_VALUE)

write_mem(os.path.join(SAVE_DIR, "stimulus.mem"), X_stim_sc.flatten())
pd.DataFrame(labels_c, columns=["Label"]).to_csv(
    os.path.join(SAVE_DIR, "expected_labels.csv"), index=False
)

n_atk = (labels_c != "BENIGN").sum()
n_ben = (labels_c == "BENIGN").sum()
print(f"✅ stimulus.mem : {X_stim_c.shape[0]} vectors  ({n_atk} attack, {n_ben} benign)")
print(f"✅ expected_labels.csv : {len(labels_c)} rows")
print(f"\n   Verilog testbench constants:")
print(f"   localparam N_VEC  = {X_stim_c.shape[0]};")
print(f"   localparam N_FEAT = {X_stim_c.shape[1]};")


📂 Building Verilog stimulus from: KDDTest+.csv
✅ stimulus.mem : 22544 vectors  (12833 attack, 9711 benign)
✅ expected_labels.csv : 22544 rows

   Verilog testbench constants:
   localparam N_VEC  = 22544;
   localparam N_FEAT = 12;


In [ ]:
from google.colab import files

print("\n📥 Downloading ...")
for name in ["mean.mem", "invstd.mem", "pca_w.mem", "threshold.mem",
             "scaler.pkl", "pca.pkl", "threshold.txt",
             "stimulus.mem", "expected_labels.csv"]:
    fpath = os.path.join(SAVE_DIR, name)
    if os.path.exists(fpath):
        files.download(fpath)
    else:
        print(f"  ⚠️  {name} not found — check earlier cells ran successfully")


📥 Downloading ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>